**TLDR**: Prepares the repository for submission against the brief's checklist: completes the package configuration, checks that every public function and class has a docstring, generates the HTML API documentation with pdoc and the HTML test report with pytest-html, adds a GitHub Actions workflow that runs the tests on every push, writes the final README from the saved results, tags a `v1.0` release and builds the zipped clone for Moodle. Work is done on the `feature/documentation` branch.


## Set-up


In [1]:
from google.colab import drive, userdata
drive.mount('/content/drive')

REPO_DIR = "/content/drive/MyDrive/MMA3001 Project/Pork-Tray-Inspection"
ZIP_PATH = "/content/drive/MyDrive/MMA3001/Pork Rasher Error-Packaging-.v4i.coco.zip"
%cd "{REPO_DIR}"

token = userdata.get("GITHUB_TOKEN")
GITHUB_USER, REPO_NAME = "chrixxv1", "Pork-Tray-Inspection"
PUSH_URL = f"https://{token}@github.com/{GITHUB_USER}/{REPO_NAME}.git"

!pip install -q pytest-html pdoc
!pip install -q -e .
import sys
sys.path.insert(0, f"{REPO_DIR}/src")
!git branch


Mounted at /content/drive
/content/drive/MyDrive/MMA3001 Project/Pork-Tray-Inspection
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for porkqc (pyproject.toml) ... done
  feature/data-loader
  feature/experiment
  feature/features
  feature/logistic
  feature/optimisation
  feature/pca
* main


## Branching


In [2]:
!git switch main
!git pull --no-rebase --no-edit {PUSH_URL} main
!git switch -c feature/documentation


M	notebooks/6_experiment.ipynb
M	notebooks/7_optimisation.ipynb
Already on 'main'
Your branch is ahead of 'origin/main' by 29 commits.
  (use "git push" to publish your local commits)
remote: Enumerating objects: 5, done.
remote: Counting objects: 100% (5/5), done.
remote: Compressing objects: 100% (3/3), done.
remote: Total 3 (delta 2), reused 0 (delta 0), pack-reused 0 (from 0)
Unpacking objects: 100% (3/3), 936 bytes | 1024 bytes/s, done.
From https://github.com/chrixxv1/Pork-Tray-Inspection
 * branch            main       -> FETCH_HEAD
Updating 2680f1e..f1d9f0a
Fast-forward
 AI_USE_LOG.md | 2 +-
 1 file changed, 1 insertion(+), 1 deletion(-)
Switched to a new branch 'feature/documentation'


## Package Configuration

Adds `matplotlib` (used by the notebooks) and the development tools (`pytest`, `pytest-html`, `pdoc`) so that `pip install -e ".[dev]"` installs everything needed to reproduce the work.


In [3]:
%%writefile pyproject.toml
[build-system]
requires = ["setuptools>=68"]
build-backend = "setuptools.build_meta"

[project]
name = "porkqc"
version = "1.0.0"
description = "Tray-level pass/fail inspection of packaged pork rashers using HOG features, PCA via the SVD and logistic regression solved by Newton's method"
readme = "README.md"
requires-python = ">=3.10"
dependencies = ["numpy", "scipy", "pillow", "scikit-image", "scikit-learn", "matplotlib"]

[project.optional-dependencies]
dev = ["pytest", "pytest-html", "pdoc"]

[tool.setuptools.packages.find]
where = ["src"]

[tool.pytest.ini_options]
testpaths = ["tests"]


Overwriting pyproject.toml


## Package Overview (src/porkqc/__init__.py)

This docstring becomes the landing page of the HTML documentation.


In [4]:
%%writefile src/porkqc/__init__.py
"""porkqc - tray-level pass/fail inspection of packaged pork rashers.

Given an overhead RGB image of a sealed tray, the package estimates the
probability that the tray has a packaging or product-placement defect and
makes a PASS/FAIL decision:

    image (720 x 540 RGB) -> HOG features (5040) -> PCA via the SVD (k = 50)
    -> logistic regression solved by Newton's method -> P(FAIL) -> threshold

Modules:

* `porkqc.data` - COCO loading, the PASS/FAIL label and input validation
* `porkqc.features` - pixel and HOG feature extraction
* `porkqc.pca` - PCA via the thin SVD and the randomized SVD
* `porkqc.logistic` - logistic regression: Newton's method and gradient descent
* `porkqc.model` - the combined PCA + logistic-regression pipeline
* `porkqc.evaluate` - metrics, threshold selection and bootstrap intervals
* `porkqc.selection` - grouped cross-validation for model selection
* `porkqc.perf` - timing, FLOP and arithmetic-intensity estimates, fast decoding
"""

__version__ = "1.0.0"


Overwriting src/porkqc/__init__.py


In [5]:
!pip install -q -e .


  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for porkqc (pyproject.toml) ... done


## Licence Check

The MIT licence was created with the repository on GitHub. This cell confirms it is present and names you as the copyright holder.


In [6]:
from pathlib import Path
lic = Path("LICENSE")
print("LICENSE present:", lic.is_file())
if lic.is_file():
    print(lic.read_text().splitlines()[0])
    print(next((l for l in lic.read_text().splitlines() if "Copyright" in l), "No copyright line found"))


LICENSE present: True
MIT License
Copyright (c) 2026 chris


## Docstring Check

Lists every public module, class, function and method in `src/porkqc` that has no docstring. An empty list means the HTML documentation covers the whole package.


In [7]:
import importlib, inspect, pkgutil
import porkqc

missing, checked = [], 0
for info in pkgutil.iter_modules(porkqc.__path__):
    mod = importlib.import_module(f"porkqc.{info.name}")
    checked += 1
    if not inspect.getdoc(mod):
        missing.append(mod.__name__)
    for name, obj in vars(mod).items():
        if name.startswith("_") or getattr(obj, "__module__", None) != mod.__name__:
            continue
        if inspect.isfunction(obj) or inspect.isclass(obj):
            checked += 1
            if not obj.__doc__:
                missing.append(f"{mod.__name__}.{name}")
        if inspect.isclass(obj):
            for mname, meth in vars(obj).items():
                if not mname.startswith("_") and inspect.isfunction(meth):
                    checked += 1
                    if not meth.__doc__:
                        missing.append(f"{mod.__name__}.{name}.{mname}")
print(f"Checked {checked} public objects; missing docstrings: {missing if missing else 'none'}")


Checked 46 public objects; missing docstrings: none


## Run the Tests and Generate the Test Report

`pytest-html` writes a self-contained HTML report, `reports/test_report.html`, listing every test and its result.


In [8]:
!mkdir -p reports
!pytest -q --html=reports/test_report.html --self-contained-html
import subprocess
collected = subprocess.run(["pytest", "--collect-only", "-q"], capture_output=True, text=True).stdout
N_TESTS = int(collected.strip().splitlines()[-1].split()[0])
print("Tests collected:", N_TESTS)


.........................................................                [100%]
- Generated html report: file:///content/drive/MyDrive/MMA3001%20Project/Pork-Tray-Inspection/reports/test_report.html -
57 passed in 9.60s
Tests collected: 57


## Generate the HTML Documentation

pdoc reads the docstrings (Google style) and writes one page per module to `docs/api/`. Open `docs/api/index.html` from the file browser, or on GitHub Pages once it is enabled.


In [9]:
!rm -rf docs/api
!pdoc porkqc -o docs/api --docformat google
!ls docs/api docs/api/porkqc


docs/api:
index.html  porkqc  porkqc.html  search.js

docs/api/porkqc:
data.html      features.html  model.html  perf.html
evaluate.html  logistic.html  pca.html	  selection.html


## Continuous Integration (GitHub Actions)

This workflow runs the tests on GitHub's servers after every push and pull request, and keeps the test report as a downloadable artifact. The tests use small synthetic data, so they run without the dataset. Pushing a workflow file needs a token with the **Workflows** permission: if the push is rejected with a "workflow" error, edit your fine-grained token on GitHub (Settings → Developer settings → your token → Repository permissions → **Workflows: Read and write**), or skip this cell.


In [10]:
!mkdir -p .github/workflows


In [11]:
%%writefile .github/workflows/tests.yml
name: tests

on:
  push:
  pull_request:

jobs:
  test:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-python@v5
        with:
          python-version: "3.12"
      - run: pip install -e ".[dev]"
      - run: pytest --html=reports/test_report.html --self-contained-html
      - uses: actions/upload-artifact@v4
        if: always()
        with:
          name: test-report
          path: reports/test_report.html


Writing .github/workflows/tests.yml


## .gitignore

Keeps the dataset, the feature cache and generated Python files out of the repository.


In [12]:
%%writefile .gitignore
# Dataset is not committed (CC BY 4.0, ~240 MB; see README)
data/*
!data/README.md
# Extracted-feature cache (regenerated by the notebooks)
results/cache/
__pycache__/
*.egg-info/
.pytest_cache/
.ipynb_checkpoints/
.venv/


Overwriting .gitignore


## Documentation Index (docs/README.md)

A landing page for the `docs/` folder that links the phase summaries and the API documentation.


In [13]:
%%writefile docs/README.md
# Documentation

## Phase summaries

| Phase | Summary |
|---|---|
| 1. Data exploration | [1_exploring_data.md](1_exploring_data.md) |
| 5. Logistic regression and solver verification | [5_logistic_regression.md](5_logistic_regression.md) |
| 6. Model selection and final evaluation | [6_experiment.md](6_experiment.md) |
| 7. Optimisation and performance | [7_optimisation.md](7_optimisation.md) |

Phases 2-4 (data loader, features, PCA) are documented by the docstrings in the API documentation.

## API documentation

Generated from the docstrings with pdoc: [api/index.html](api/index.html).
Each module page lists the functions and classes with their arguments, return values and exceptions.


Writing docs/README.md


## README

Written from the saved results (`results/6_metrics.json`, `results/7_profile.json`) so the numbers match your runs. Check the generated text below, especially the timing figures, which depend on the Colab machine used in notebook 7.


In [14]:
import json
m6 = json.load(open("results/6_metrics.json"))
m7 = json.load(open("results/7_profile.json"))
t, sel = m6["test"], m6["selected"]
lo, hi = m6["test_auc_ci95"]
rlo, rhi = m6["test_recall_ci95"]
rbd = m6["recall_by_defect"]
defects = ", ".join(f"{c} {v['recall']:.2f} (n = {v['n']})" for c, v in sorted(rbd.items(), key=lambda kv: -kv[1]["n"]))
expo = [v["auc"] for v in m6["exposure_robustness"].values()]
mlp = ", ".join(f"{a:.3f}" for a in m6["mlp"]["test_auc_seeds"])
ls, ld = m7["latency_standard"], m7["latency_draft"]
pf, pr = m7["pca"]["full"], m7["pca"]["randomized_q10"]
s400 = {r["solver"]: r for r in m7["solvers"] if r["k"] == 400}

readme = f"""# Pork-Tray-Inspection

MMA3001 Numerical Methods and Machine Learning - individual project, Monash University.

Automated **pass/fail inspection of packaged pork-rasher trays** from overhead RGB images, built from
numerical methods: HOG features, PCA via the singular value decomposition, and L2-regularised logistic
regression solved by Newton's method. Gradient descent, the randomized SVD, pixel features and a
neural network are compared as alternatives.

```
image (720 x 540 RGB) -> HOG (5040) -> PCA via SVD (k = {sel['k']}) -> logistic regression (Newton) -> P(FAIL) -> threshold
```

## Engineering problem and scope

Sealed trays leave a packing line faster than operators can inspect them consistently. A tray with an
unsealed film or misplaced meat can compromise food safety and shelf life. This project supports
automated inspection by flagging suspect trays for a manual check. It addresses the **tray-level
decision only**: it does not localise defects, identify their type or estimate their severity, because
the rarer defect classes have too few validation and test examples (3-4 each, and none for wrinkle)
to evaluate per-class detection.

## Inputs and outputs

| | Description |
|---|---|
| **Input** | One overhead RGB image of a tray, JPEG, 720 x 540 px, 8-bit (0-255). Other sizes are processed but raise a warning, because they lie outside the validated domain. |
| **Training labels** | COCO bounding boxes; an image is FAIL (1) if it has any annotation, otherwise PASS (0). |
| **Output 1** | P(FAIL), a probability in [0, 1]. |
| **Output 2** | PASS/FAIL from a threshold of {m6['threshold']:.3f}, chosen on the validation split for {m6['target_recall']:.0%} recall. FAIL trays are diverted for manual inspection. |
| **Invalid input** | Missing files raise `FileNotFoundError`; malformed COCO files, wrong array shapes, values outside [0, 1], NaN/inf and labels other than 0/1 raise `ValueError`. |

Function-level inputs and outputs are documented in the docstrings and the HTML documentation.

## Installation

**Google Colab (as used for this project).** Open `notebooks/0_setup.ipynb` and follow its instructions:
it mounts Google Drive, clones this repository, unzips the dataset and installs the package.

**Local machine (Python 3.10+):**

```bash
git clone https://github.com/{GITHUB_USER}/{REPO_NAME}.git
cd {REPO_NAME}
python -m venv .venv
source .venv/bin/activate        # Windows: .venv\\Scripts\\activate
pip install -e ".[dev]"
```

## Data

The dataset is **not** included in this repository. Download *Pork Rasher Error (Packaging) v4* in **COCO**
format from Roboflow Universe (licence CC BY 4.0) and unzip it so that `train/`, `valid/` and `test/`
each contain `_annotations.coco.json`. See `data/README.md`.

## Usage

Run the notebooks in order; each one documents its own steps and expected outputs.

| Notebook | Purpose |
|---|---|
| `0_setup.ipynb` | Colab, Google Drive and GitHub set-up |
| `1_exploring_data.ipynb` | Dataset exploration, annotation checks, scope |
| `2_data_module.ipynb` | Data loader and PASS/FAIL label |
| `3_features.ipynb` | Pixel and HOG features |
| `4_pca.ipynb` | PCA via the SVD |
| `5_logistic.ipynb` | Logistic regression: Newton's method and gradient descent |
| `6_experiment.ipynb` | Grouped cross-validation, model selection, final test evaluation |
| `7_optimisation.ipynb` | Profiling, timing, FLOPs, arithmetic intensity, roofline |
| `8_documentation.ipynb` | Documentation, test report and submission files |

Run the tests and rebuild the documentation:

```bash
pytest --html=reports/test_report.html --self-contained-html
pdoc porkqc -o docs/api --docformat google
```

## Results (test split: 80 images, 68 FAIL and 12 PASS, evaluated once)

Selected by grouped 5-fold cross-validation: HOG features, k = {sel['k']} components, lambda = {sel['l2']:g}
(CV AUC {sel['cv_auc']:.3f}).

| Metric | Model | Always-FAIL baseline |
|---|---|---|
| AUC (95 % bootstrap CI) | {t['auc']:.3f} ({lo:.2f}-{hi:.2f}) | 0.500 |
| Recall | {t['recall']:.3f} ({rlo:.2f}-{rhi:.2f}) | 1.000 |
| Specificity | {t['specificity']:.3f} | 0.000 |
| Precision | {t['precision']:.3f} | {m6['baseline_always_fail']['precision']:.3f} |
| Balanced accuracy | {t['balanced_accuracy']:.3f} | 0.500 |

- Recall by defect type: {defects}.
- Neural-network (MLP) alternative: test AUC {mlp} over three random seeds.
- Robust to exposure changes of -30 % to +30 %: test AUC {min(expo):.3f}-{max(expo):.3f}.
- The 90 % recall target on validation gave {t['recall']:.0%} on test: the threshold does not transfer
  reliably from 106 FAIL validation images.

Performance (Colab CPU; see `docs/7_optimisation.md`):

- Inference takes {ls['total_ms']:.1f} ms per image ({ls['images_per_s']:.0f} images/s); reduced-resolution
  JPEG decoding cuts this to {ld['total_ms']:.1f} ms ({ld['images_per_s']:.0f} images/s) with no loss of accuracy.
- PCA: the randomized SVD (10 power iterations) takes {pr['time_s']:.1f} s against {pf['time_s']:.1f} s for the full SVD,
  with the same subspace.
- Training at k = 400: Newton's method needs {s400['newton']['iterations']} iterations ({1e3 * s400['newton']['time_s']:.0f} ms);
  gradient descent needs {s400['gd']['iterations']} ({1e3 * s400['gd']['time_s']:.0f} ms).

## Verification and testing

{N_TESTS} pytest tests (`tests/`, report in `reports/test_report.html`), including gradient and Hessian
checks against central finite differences, agreement with scikit-learn (PCA and logistic regression),
quadratic convergence of Newton's method, the explicit-Euler stability limit of gradient descent,
overflow-safe sigmoid and loss, grouped cross-validation leakage, and input validation.
They use small synthetic data, so they run without the dataset.

## Repository structure

```
src/porkqc/      package source (data, features, pca, logistic, model, evaluate, selection, perf)
tests/           pytest suite
notebooks/       Colab notebooks for each phase
docs/            phase summaries; docs/api/ holds the HTML API documentation
results/         metrics (JSON), profiling output and figures
reports/         pytest HTML test report
data/            dataset location (not committed)
```

## Licence

Code: MIT (see `LICENSE`). Dataset: CC BY 4.0, from Roboflow Universe; not redistributed here.

## AI use

AI assistance (Claude) was used throughout this project. Every use is recorded in `AI_USE_LOG.md`,
and the report contains the AI-use reflection.
"""
with open("README.md", "w") as f:
    f.write(readme)
print(readme)


# Pork-Tray-Inspection

MMA3001 Numerical Methods and Machine Learning - individual project, Monash University.

Automated **pass/fail inspection of packaged pork-rasher trays** from overhead RGB images, built from
numerical methods: HOG features, PCA via the singular value decomposition, and L2-regularised logistic
regression solved by Newton's method. Gradient descent, the randomized SVD, pixel features and a
neural network are compared as alternatives.

```
image (720 x 540 RGB) -> HOG (5040) -> PCA via SVD (k = 50) -> logistic regression (Newton) -> P(FAIL) -> threshold
```

## Engineering problem and scope

Sealed trays leave a packing line faster than operators can inspect them consistently. A tray with an
unsealed film or misplaced meat can compromise food safety and shelf life. This project supports
automated inspection by flagging suspect trays for a manual check. It addresses the **tray-level
decision only**: it does not localise defects, identify their type or estimate their

## Repository Check

Lists the tracked files and flags any file larger than 50 MB (GitHub rejects files over 100 MB).


In [15]:
!git add -A --dry-run | head -50
import os
big = [(p, os.path.getsize(p) / 1e6) for p in
       os.popen("git ls-files; git ls-files --others --exclude-standard").read().split() if os.path.isfile(p)
       and os.path.getsize(p) > 50e6]
print("Files over 50 MB:", big if big else "none")


add '.gitignore'
add 'README.md'
add 'notebooks/6_experiment.ipynb'
add 'notebooks/7_optimisation.ipynb'
add 'pyproject.toml'
add 'src/porkqc/__init__.py'
add '.github/workflows/tests.yml'
add 'docs/README.md'
add 'docs/api/index.html'
add 'docs/api/porkqc.html'
add 'docs/api/porkqc/data.html'
add 'docs/api/porkqc/evaluate.html'
add 'docs/api/porkqc/features.html'
add 'docs/api/porkqc/logistic.html'
add 'docs/api/porkqc/model.html'
add 'docs/api/porkqc/pca.html'
add 'docs/api/porkqc/perf.html'
add 'docs/api/porkqc/selection.html'
add 'docs/api/search.js'
add 'notebooks/8_documentation.ipynb'
add 'reports/test_report.html'
Files over 50 MB: none


## AI Usage Log

**Run once only**, then put `#` in front of each line.


In [16]:
%%writefile -a AI_USE_LOG.md
| 2026-10-27 | Claude | Wrote the code and summaries: package configuration, docstring check, pytest-html test report, pdoc HTML documentation, GitHub Actions workflow, documentation index and the README generated from my saved results | Checked the README numbers against my Phase 6 and 7 outputs, opened the HTML documentation and test report, and confirmed the repository contents on GitHub |


Appending to AI_USE_LOG.md


## Commit


In [17]:
!git add -A
!git status --short
!git commit -m "Add final README, HTML API docs, test report, CI workflow and packaging for v1.0"
!git push {PUSH_URL} feature/documentation


A  .github/workflows/tests.yml
M  .gitignore
M  AI_USE_LOG.md
M  README.md
A  docs/README.md
A  docs/api/index.html
A  docs/api/porkqc.html
A  docs/api/porkqc/data.html
A  docs/api/porkqc/evaluate.html
A  docs/api/porkqc/features.html
A  docs/api/porkqc/logistic.html
A  docs/api/porkqc/model.html
A  docs/api/porkqc/pca.html
A  docs/api/porkqc/perf.html
A  docs/api/porkqc/selection.html
A  docs/api/search.js
M  notebooks/6_experiment.ipynb
M  notebooks/7_optimisation.ipynb
A  notebooks/8_documentation.ipynb
M  pyproject.toml
A  reports/test_report.html
M  src/porkqc/__init__.py
[feature/documentation 3d0c080] Add final README, HTML API docs, test report, CI workflow and packaging for v1.0
 22 files changed, 7865 insertions(+), 9 deletions(-)
 create mode 100644 .github/workflows/tests.yml
 create mode 100644 docs/README.md
 create mode 100644 docs/api/index.html
 create mode 100644 docs/api/porkqc.html
 create mode 100644 docs/api/porkqc/data.html
 create mode 100644 docs/api/porkqc/eva

In [ ]:
!git pull --no-rebase --no-edit {PUSH_URL} main
!git branch -f main HEAD
!git switch main
!git log --oneline --graph -5


## Tag the v1.0 Release

A tag marks the exact commit you submit. It appears under **Releases/Tags** on GitHub.


In [ ]:
!git tag -a v1.0 -m "Submitted version"
!git push {PUSH_URL} v1.0


## GitHub Pages (optional, on github.com)

To give markers a live link to the HTML documentation: on your repository page go to **Settings** → **Pages** → Source **Deploy from a branch** → branch `main`, folder `/docs` → **Save**. After a minute the documentation is at `https://chrixxv1.github.io/Pork-Tray-Inspection/api/`.

**Repository access**: the brief requires an *accessible* link. If your repository is private, either make it public (Settings → General → Danger Zone → Change visibility) or add the teaching team as collaborators (Settings → Collaborators). Check edStem for which the unit expects. GitHub Pages on a private repository needs a paid plan.


## Zipped Clone for Moodle

Clones the repository fresh from GitHub (so the zip is exactly what markers see, including the full commit history in `.git`), removes the token from the clone's settings, zips it and saves the zip to your Drive project folder for download.


In [ ]:
!rm -rf /content/submission && mkdir -p /content/submission
!git clone -q {PUSH_URL} /content/submission/{REPO_NAME}
# The clone stores the URL it came from; replace it so the token is not inside the zip.
!git -C /content/submission/{REPO_NAME} remote set-url origin https://github.com/{GITHUB_USER}/{REPO_NAME}.git
!git -C /content/submission/{REPO_NAME} remote -v
!git -C /content/submission/{REPO_NAME} log --oneline | wc -l
!cd /content/submission && zip -qr {REPO_NAME}.zip {REPO_NAME}
!cp /content/submission/{REPO_NAME}.zip "/content/drive/MyDrive/MMA3001 Project/"
!ls -lh "/content/drive/MyDrive/MMA3001 Project/{REPO_NAME}.zip"


The `remote -v` output must show `https://github.com/chrixxv1/Pork-Tray-Inspection.git` with **no token** in it. The commit count shows the history is included. Download the zip from your Drive `MMA3001 Project` folder and submit it on Moodle, together with the repository link.
